# OTIS Application 1: Uvas Creek Tracer Experiment

This notebook demonstrates the `RiverSoluteTransportDynamics` Landlab component by replicating **Application 1** from the USGS OTIS (One-Dimensional Transport with Inflow and Storage) user manual.

## Background
In 1983, a 3-hour continuous injection of chloride (a conservative tracer) was performed on a 669-meter reach of Uvas Creek, California. The experiment highlighted the critical role of **transient storage** (hyporheic exchange and dead zones) in riverine solute transport. Standard advection-dispersion models fail to capture the long "tailing" effect seen in the Uvas Creek breakthrough curves.

In this tutorial, we will:
1. Set up a 1D-equivalent Landlab grid.
2. Initialize steady-state hydraulics.
3. Parameterize the `RiverSoluteTransportDynamics` component with transient storage.
4. Simulate the 3-hour step injection and plot the downstream breakthrough curves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from landlab import RasterModelGrid
from landlab.components import RiverSoluteTransportDynamics

# Plotting configuration
%matplotlib inline
plt.rcParams.update({'font.size': 12})

## 1. Grid and Hydraulic Setup
Uvas Creek is simulated over a 669 m reach. Since our Landlab component solves the 2D depth-averaged equations, we will create a narrow 3-row grid and close the lateral boundaries to simulate a 1D channel down the center. 

We will use average hydraulic parameters derived from the OTIS documentation.

In [ ]:
# 2. Grid Setup and Spatially Variable Hydraulics
dx = 10.0
nx = 68  
grid = RasterModelGrid((3, nx), xy_spacing=dx)

grid.status_at_node[grid.nodes_at_top_edge] = grid.BC_NODE_IS_CLOSED
grid.status_at_node[grid.nodes_at_bottom_edge] = grid.BC_NODE_IS_CLOSED

Q = 0.0125       
width = 1.5      

# Create 1D arrays for the 5 distinct OTIS Reaches
depth_1d = np.zeros(nx)
vel_1d = np.zeros(nx)
alpha_1d = np.zeros(nx)
hs_1d = np.ones(nx) * 0.01 # Small default to avoid div by zero

def set_reach(start_col, end_col, A, A_s, alpha):
    depth_1d[start_col:end_col] = A / width
    vel_1d[start_col:end_col] = Q / A
    alpha_1d[start_col:end_col] = alpha
    if A_s > 0:
        hs_1d[start_col:end_col] = A_s / width

# Reach 1 (0-38m)
set_reach(0, 4, A=0.58, A_s=0.0, alpha=0.0)
# Reach 2 (38-105m)
set_reach(4, 11, A=0.65, A_s=0.0, alpha=0.0)
# Reach 3 (105-281m) - Massive storage begins here
set_reach(11, 29, A=0.81, A_s=0.6, alpha=3.4e-5)
# Reach 4 (281-433m)
set_reach(29, 44, A=0.70, A_s=0.6, alpha=1.6e-5)
# Reach 5 (433-619m)
set_reach(44, nx, A=0.70, A_s=0.4, alpha=1.0e-5)

# Broadcast the 1D arrays across the 3 rows of the Landlab grid
grid.add_zeros("surface_water__depth", at="node")[:] = np.tile(depth_1d, (3, 1)).flatten()

# Initialize velocity fields
grid.add_zeros("surface_water__velocity", at="link")
grid.add_zeros("advection__velocity", at="link")

# Tile the 1D velocities 3 times to cover all 3 rows (3 * 67 = 201 links)
vel_grid_horizontal = np.tile(vel_1d[:-1], 3)

# Assign the tiled array to the horizontal links
grid.at_link["surface_water__velocity"][grid.horizontal_links] = vel_grid_horizontal
grid.at_link["advection__velocity"][grid.horizontal_links] = vel_grid_horizontal

# Store the storage parameters as full grid arrays to pass to the component
alpha_grid = np.tile(alpha_1d, (3, 1)).flatten()
hs_grid = np.tile(hs_1d, (3, 1)).flatten()

## 2. Component Initialization
We initialize the `RiverSoluteTransportDynamics` component for a single conservative solute (`Chloride`). 

The crucial OTIS parameters here are the storage zone cross-sectional area (expressed here as an equivalent depth, `h_storage`) and the mass exchange coefficient (`alpha_exchange`). Because Chloride is conservative, we do not provide decay or sorption parameters.

In [ ]:
# 3. Component Initialization (Spatially Variable)
solutes = ["Chloride"]

rstd = RiverSoluteTransportDynamics(
    grid,
    solutes=solutes,
    alpha_L=2.0,            
    # Pass the spatially variable arrays instead of constants!
    alpha_exchange={"Chloride": alpha_grid},  
    h_storage={"Chloride": hs_grid},         
    outlet_boundary_condition="zero_gradient"
)

## 3. Boundary Conditions and Simulation Loop
The experiment consisted of a step injection. The background concentration of Chloride was 4.5 mg/L. For 3 hours, the upstream boundary concentration was raised to 12.0 mg/L, after which it was returned to background levels. We will run the simulation for a total of 6 hours to capture the recession limb (the "tail").

In [ ]:
background_C = 3.7  # mg/L
injection_C = 11.4  # mg/L

grid.at_node["surface_water__Chloride__concentration"][:] = background_C
grid.at_node["storage_zone__Chloride__concentration"][:] = background_C

# Observation stations: 38m, 105m, 281m, and 433m
obs_nodes = [72, 79, 97, 112] 
labels = ["38 m", "105 m", "281 m", "433 m"]

times = []
obs_data = {node: [] for node in obs_nodes}

# Record the exact starting state (t = 0)
times.append(0.0)
for node in obs_nodes:
    obs_data[node].append(grid.at_node["surface_water__Chloride__concentration"][node])

# --- MAIN TIME LOOP ---
dt = 5.0
t_max = 3600 * 30
injection_duration = 3600 * 3

for t in range(int(dt), t_max + int(dt), int(dt)):
    inflow_nodes = grid.nodes_at_left_edge
    
    # Step injection logic
    if t <= injection_duration:
        grid.at_node["surface_water__Chloride__concentration"][inflow_nodes] = injection_C
    else:
        grid.at_node["surface_water__Chloride__concentration"][inflow_nodes] = background_C
    
    rstd.run_one_step(dt)
    
    times.append(t / 3600.0)
    for node in obs_nodes:
        obs_data[node].append(grid.at_node["surface_water__Chloride__concentration"][node])


## 4. Results
The plotted breakthrough curves show the classic hallmarks of transient storage. Note the long, asymmetric tail on the downstream observation points (281 m and 433 m) after the injection stops at hour 3. This slow release of mass from the hyporheic zone is exactly what OTIS was designed to capture.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']

for idx, node in enumerate(obs_nodes):
    ax.plot(times, obs_data[node], label=f"Station {labels[idx]}", color=colors[idx], linewidth=2)

ax.axvspan(0, 3, color='gray', alpha=0.15, label='Injection Period')
ax.set_title("Uvas Creek: Conservative Tracer (Spatially Variable Hydraulics)", fontweight='bold')
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Concentration (mg/L)")
ax.set_ylim(2.0, 12.5) # Force the axes to show the background tail clearly
ax.legend(loc="upper right")
ax.grid(True, linestyle='--', alpha=0.7)
plt.show()